# Visualize Metadata Subgroups for SNPs 
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import defaultdict
from itertools import product
import re
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import seaborn as sns

from hk1_r12ter_analysis.config import (
    GENE_ID,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    RAW_DATA_DIR,
    REPORTS_DIR,
    logger,
)
from hk1_r12ter_analysis.io import load_dataframe_from_file, make_filename
from hk1_r12ter_analysis.linkage_disequilibrium import identify_independent_snps
from hk1_r12ter_analysis.util import ceil_to_decimal

## REDS Index

In [ ]:
sample_key = "RECALL DONOR ID"
group_key = "Day"
source_type = "REDS-Recall"
main_data_type = f"Genomics-{GENE_ID}"
other_data_type = "Metadata"

correlation_statistic = "Spearman"
r2_threshold = 0.8
prioritize_genotyped_in_LD = True

filetype = "pdf"
variable_fancy_replacements = {
    f"{source_type.split('-')[-1]}.Adjusted.Osmotic.Hemolysis": "Recall Osmotic Hemolysis",
    f"{source_type.split('-')[-1]}.Adjusted.Storage.Hemolysis": "Recall Storage Hemolysis",
    f"{source_type.split('-')[-1]}.Adjusted.Oxidative.Hemolysis": "Recall Oxidative Hemolysis",
    source_type: source_type.replace("-", " "),
}

### Load numerical metadata and genomics

In [ ]:
# Load metadata + genomics
input_dirpath = PROCESSED_DATA_DIR / "REDS"
filename = make_filename(source_type, "Donor", f"Genomics-{GENE_ID}_Metadata")
df_metadata_genomics = load_dataframe_from_file(
    input_dirpath / filename, index_col=sample_key, header=[0, 1]
)
df_metadata_genomics = df_metadata_genomics.convert_dtypes(convert_integer=False)

metadata_columns = df_metadata_genomics.loc[:, "Metadata"].columns.tolist()
# Filter for genotyped SNPs only if desired
genomic_columns = df_metadata_genomics.loc[:, f"Genomics-{GENE_ID}"].columns.tolist()
df_metadata_genomics

### Load genomics metadata

In [ ]:
input_dirpath = PROCESSED_DATA_DIR / "REDS"
input_filename = make_filename("REDS", "Genomics", GENE_ID, "Metadata")
df_genomics_ann = load_dataframe_from_file(input_dirpath / input_filename, index_col="ID")
df_genomics_ann = df_genomics_ann.loc[genomic_columns]
df_genomics_ann = df_genomics_ann.drop("ANN_Priority", axis=1)
df_rsid_map = df_genomics_ann["RSID"].replace(":", ".")
df_genomics_ann

### Load correlation and statistics

In [ ]:
index_cols = ["Group", "DataType1", "Variable1", "DataType2", "Variable2"]
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
input_filename = make_filename(source_type, f"{main_data_type}_All_Data", "Statistics")

df_correlations_tall = load_dataframe_from_file(
    input_dirpath / input_filename, index_col=None, low_memory=False
)

group_value = "ALL"
correlations_options = {
    "Group": group_value,
    "DataType1": main_data_type,
    "DataType2": other_data_type,
}
for key, value in correlations_options.items():
    df_correlations_tall = (
        df_correlations_tall[df_correlations_tall[key] == value]
        .drop_duplicates()
        .drop(key, axis=1)
    )
df_correlations_tall = df_correlations_tall.set_index(["Variable1", "Variable2"])
df_correlations_tall.index.names = ["ID", "Variable"]
df_correlations_tall = df_correlations_tall.swaplevel(0, 1)
df_correlations_tall

#### Load linkage disequilibrium $r^{2}$ correlation matrices

In [ ]:
LD_method = "RH"
rsids = df_genomics_ann.index.tolist()
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
input_filename = make_filename(
    "REDS-Index",  # Always REDS-Index cohort
    f"Genomics-{GENE_ID}_Metadata",
    LD_method,
    "LinkageDisequilibrium",
    "ALL",
)
df_ld_matrix = load_dataframe_from_file(input_dirpath / input_filename, index_col=["SNP1", "SNP2"])
df_r2_matrix = df_ld_matrix.loc[:, "r2"]
df_r2_matrix

### Breakdown SNP by subgroup

In [ ]:
variable = f"{source_type.split('-')[-1]}.Adjusted.Osmotic.Hemolysis"
variable_fancy = variable_fancy_replacements.get(variable, variable)
df = df_correlations_tall.loc[variable]
rsids = df.index
df = df.loc[rsids].sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False)
df = df.join(df_genomics_ann, how="left")
cols_of_interest = [
    "RSID",
    "Spearman statistic",
    "Spearman -log10(p-value)",
    "ANN_Annotation_Impact",
    "ANN_Gene_Name",
    "ANN_Feature_Type",
    "ANN_Transcript_BioType",
    "ANN_HGVS.c",
    "ANN_HGVS.p",
]
if prioritize_genotyped_in_LD:
    df.loc[:, "Type"] = df.reset_index(drop=False)["ID"].str.split("_", n=1, expand=True)[0].values
    df = df.sort_values(
        by=["Type", f"{correlation_statistic} -log10(p-value)"], ascending=[True, False]
    ).drop("Type", axis=1)
else:
    df = df.loc[variable].sort_values(
        by=f"{correlation_statistic} -log10(p-value)", ascending=False
    )
rsids = identify_independent_snps(
    df_r2_matrix=df_r2_matrix,
    list_of_snps=df.index.tolist(),
    r2_threshold=r2_threshold,
)
df = df.loc[rsids, cols_of_interest].sort_values(
    by=f"{correlation_statistic} -log10(p-value)", ascending=False
)
rsids = df.index.tolist()
df

In [ ]:
include_table = False
grid_kwargs = dict(
    linestyle="-",
    color="xkcd:light grey",
    which="both",
)
legend_kwargs = dict(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.25),
    edgecolor="black",
    ncols=2,
    fontsize="medium",
)
bar_width = 0.6
allele_name_map = {0: "Homozygous Reference", 1: "Heterozygous", 2: "Homozygous Alternate"}
ylabel = None
label_fontdict = {"size": "large"}
title_fontdict = {"size": "x-large", "weight": "bold"}
tick_step = 0.2

### Sex

In [ ]:
subgroup = "Sex"
title = subgroup
cut_kwargs = {}
palette = {"F": mpl.colors.hex2color("#96f97b"), "M": mpl.colors.hex2color("#95d0fc")}
cmap_name = None
n_rsids = 0


output_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "Correlations"
    / f"{main_data_type}_{other_data_type}"
    / subgroup.replace(" ", "")
)
output_dirpath.mkdir(parents=True, exist_ok=True)
for idx, rsid in enumerate(rsids):
    xlabel = rsid
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
    data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
    data = data.droplevel(level=0, axis=1).dropna()

    if cut_kwargs:
        data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
    data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
    data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
    data = data.sort_values(by=[rsid, subgroup])
    data = data.pivot(columns=subgroup, index=rsid)
    data = data.fillna(0)
    subgroups = list(data["count"].columns[::-1])
    # Get the colormap and select colors from the colormap
    # Values between 0 and 1 are used to select colors along the map
    if not palette and cmap_name:
        cmap = mpl.colormaps.get_cmap(cmap_name)
        colors = [
            cmap(x)
            for x in np.linspace(
                0 if cmap_name.endswith("_r") else 0.2,
                0.8 if cmap_name.endswith("_r") else 1,
                len(subgroups),
            )
        ]
        palette = dict(zip(subgroups, colors))

    ax = data["percentage"].plot(
        kind="bar",
        use_index=True,
        ax=ax,
        stacked=True,
        zorder=4,
        width=bar_width,
        xlabel=rsid,
        ylabel=None,
        legend=False,
        color=palette,
        edgecolor="black",
    )

    ax.set_ylim(0, 1.1)
    ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
    ax.set_ylabel(ylabel, fontdict=label_fontdict)
    ax.set_title(title, fontdict=title_fontdict)
    ax.yaxis.grid(**grid_kwargs)
    ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
    ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
    if include_table:
        ax.set_xticks([])
        ax.set_xlabel(None)
        ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

        row_labels = subgroups
        col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
        cell_text = np.array(
            [row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()]
        )
        table = ax.table(
            cellText=cell_text,
            cellLoc="center",
            rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
            rowLoc="center",
            colLabels=col_labels,
            colLoc="center",
            bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
        )
        for i in range(0, len(row_labels) + 1):
            for j in range(-1, len(col_labels)):
                if i == 0 and j == -1:
                    continue
                elif i == 0:
                    color = "xkcd:black"
                else:
                    color = palette.get(row_labels[i - 1], "xkcd:black")
                cell = table[(i, j)]
                cell.set_text_props(weight="bold", color=color)
        table.auto_set_font_size(False)
        table.set_fontsize(7.5)
    else:
        ax.set_xticks(
            ax.get_xticks(),
            labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
            rotation=0,
            ha="center",
        )
        if legend_kwargs:
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(handles=handles, labels=labels, **legend_kwargs)

    sns.despine(ax=ax)

    filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
    if include_table:
        filename_args += ["table"]
    if group_value != "ALL":
        filename_args += [f"{group_key}{group_value}"]
    output_filename = make_filename(*filename_args, filetype=filetype)
    fig.savefig(output_dirpath / output_filename)
    plt.show()
    plt.close()
    print(output_filename)
    print(data["count"].sum(axis=1))
    if n_rsids <= idx:
        break

In [ ]:
rsid = rsids[0]
subgroup = "Sex"
title = subgroup
cut_kwargs = {}
palette = {"F": mpl.colors.hex2color("#96f97b"), "M": mpl.colors.hex2color("#95d0fc")}
cmap_name = None


output_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "Correlations"
    / f"{main_data_type}_{other_data_type}"
    / subgroup.replace(" ", "")
)
output_dirpath.mkdir(parents=True, exist_ok=True)
xlabel = rsid
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
data = data.droplevel(level=0, axis=1).dropna()

if cut_kwargs:
    data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
data = data.sort_values(by=[rsid, subgroup])
data = data.pivot(columns=subgroup, index=rsid)
data = data.fillna(0)
subgroups = list(data["count"].columns[::-1])
# Get the colormap and select colors from the colormap
# Values between 0 and 1 are used to select colors along the map
if not palette and cmap_name:
    cmap = mpl.colormaps.get_cmap(cmap_name)
    colors = [
        cmap(x)
        for x in np.linspace(
            0 if cmap_name.endswith("_r") else 0.2,
            0.8 if cmap_name.endswith("_r") else 1,
            len(subgroups),
        )
    ]
    palette = dict(zip(subgroups, colors))

ax = data["percentage"].plot(
    kind="bar",
    ax=ax,
    use_index=True,
    # ax=ax,
    stacked=True,
    zorder=4,
    width=bar_width,
    xlabel=rsid,
    ylabel=None,
    legend=False,
    color=palette,
    edgecolor="black",
)

ax.set_ylim(0, 1.1)
ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
ax.set_ylabel(ylabel, fontdict=label_fontdict)
ax.set_title(title, fontdict=title_fontdict)
ax.yaxis.grid(**grid_kwargs)
ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
if include_table:
    ax.set_xticks([])
    ax.set_xlabel(None)
    ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

    row_labels = subgroups
    col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
    cell_text = np.array([row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()])
    table = ax.table(
        cellText=cell_text,
        cellLoc="center",
        rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
        rowLoc="center",
        colLabels=col_labels,
        colLoc="center",
        bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
    )
    for i in range(0, len(row_labels) + 1):
        for j in range(-1, len(col_labels)):
            if i == 0 and j == -1:
                continue
            elif i == 0:
                color = "xkcd:black"
            else:
                color = palette.get(row_labels[i - 1], "xkcd:black")
            cell = table[(i, j)]
            cell.set_text_props(weight="bold", color=color)
    table.auto_set_font_size(False)
    table.set_fontsize(7.5)
else:
    ax.set_xticks(
        ax.get_xticks(),
        labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
        rotation=0,
        ha="center",
    )
    if legend_kwargs:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles=handles, labels=labels, **legend_kwargs)


sns.despine(ax=ax)

filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
if include_table:
    filename_args += ["table"]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
output_filename = make_filename(*filename_args, filetype=filetype)
fig.savefig(output_dirpath / output_filename)
plt.show()
plt.close()
print(output_filename)
print(data["count"].sum(axis=1))

#### Ethnicity

In [ ]:
subgroup = "Ethnicity"
title = subgroup
cut_kwargs = {}
palette = {}
cmap_name = "Reds"
n_rsids = 0


output_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "Correlations"
    / f"{main_data_type}_{other_data_type}"
    / subgroup.replace(" ", "")
)
output_dirpath.mkdir(parents=True, exist_ok=True)
for idx, rsid in enumerate(rsids):
    xlabel = rsid
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
    data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
    data = data.droplevel(level=0, axis=1).dropna()

    if cut_kwargs:
        data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
    data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
    data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
    data = data.sort_values(by=[rsid, subgroup])
    data = data.pivot(columns=subgroup, index=rsid)
    data = data.fillna(0)
    subgroups = list(data["count"].columns[::-1])
    # Get the colormap and select colors from the colormap
    # Values between 0 and 1 are used to select colors along the map
    if not palette and cmap_name:
        cmap = mpl.colormaps.get_cmap(cmap_name)
        colors = [
            cmap(x)
            for x in np.linspace(
                0 if cmap_name.endswith("_r") else 0.2,
                0.8 if cmap_name.endswith("_r") else 1,
                len(subgroups),
            )
        ]
        palette = dict(zip(subgroups, colors))

    ax = data["percentage"].plot(
        kind="bar",
        use_index=True,
        ax=ax,
        stacked=True,
        zorder=4,
        width=bar_width,
        xlabel=rsid,
        ylabel=None,
        legend=False,
        color=palette,
        edgecolor="black",
    )

    ax.set_ylim(0, 1.1)
    ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
    ax.set_ylabel(ylabel, fontdict=label_fontdict)
    ax.set_title(title, fontdict=title_fontdict)
    ax.yaxis.grid(**grid_kwargs)
    ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
    ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
    if include_table:
        ax.set_xticks([])
        ax.set_xlabel(None)
        ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

        row_labels = subgroups
        col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
        cell_text = np.array(
            [row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()]
        )
        table = ax.table(
            cellText=cell_text,
            cellLoc="center",
            rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
            rowLoc="center",
            colLabels=col_labels,
            colLoc="center",
            bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
        )
        for i in range(0, len(row_labels) + 1):
            for j in range(-1, len(col_labels)):
                if i == 0 and j == -1:
                    continue
                elif i == 0:
                    color = "xkcd:black"
                else:
                    color = palette.get(row_labels[i - 1], "xkcd:black")
                cell = table[(i, j)]
                cell.set_text_props(weight="bold", color=color)
        table.auto_set_font_size(False)
        table.set_fontsize(7.5)
    else:
        ax.set_xticks(
            ax.get_xticks(),
            labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
            rotation=0,
            ha="center",
        )
        if legend_kwargs:
            handles, labels = ax.get_legend_handles_labels()
            print(labels)
            ax.legend(handles=handles, labels=labels, **legend_kwargs)

    sns.despine(ax=ax)

    filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
    if include_table:
        filename_args += ["table"]
    if group_value != "ALL":
        filename_args += [f"{group_key}{group_value}"]
    output_filename = make_filename(*filename_args, filetype=filetype)
    fig.savefig(output_dirpath / output_filename)
    plt.show()
    plt.close()
    print(output_filename)
    print(data["count"].sum(axis=1))
    if n_rsids <= idx:
        break

In [ ]:
rsid = rsids[0]
subgroup = "Ethnicity"
title = subgroup
cut_kwargs = {}
palette = {}
cmap_name = "Reds"


output_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "Correlations"
    / f"{main_data_type}_{other_data_type}"
    / subgroup.replace(" ", "")
)
output_dirpath.mkdir(parents=True, exist_ok=True)
xlabel = rsid
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
data = data.droplevel(level=0, axis=1).dropna()

if cut_kwargs:
    data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
data = data.sort_values(by=[rsid, subgroup])
data = data.pivot(columns=subgroup, index=rsid)
data = data.fillna(0)
subgroups = list(data["count"].columns[::-1])
# Get the colormap and select colors from the colormap
# Values between 0 and 1 are used to select colors along the map
if not palette and cmap_name:
    cmap = mpl.colormaps.get_cmap(cmap_name)
    colors = [
        cmap(x)
        for x in np.linspace(
            0 if cmap_name.endswith("_r") else 0.2,
            0.8 if cmap_name.endswith("_r") else 1,
            len(subgroups),
        )
    ]
    palette = dict(zip(subgroups, colors))

ax = data["percentage"].plot(
    kind="bar",
    use_index=True,
    ax=ax,
    stacked=True,
    zorder=4,
    width=bar_width,
    xlabel=rsid,
    ylabel=None,
    legend=False,
    color=palette,
    edgecolor="black",
)

ax.set_ylim(0, 1.1)
ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
ax.set_ylabel(ylabel, fontdict=label_fontdict)
ax.set_title(title, fontdict=title_fontdict)
ax.yaxis.grid(**grid_kwargs)
ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
if include_table:
    ax.set_xticks([])
    ax.set_xlabel(None)
    ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

    row_labels = subgroups
    col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
    cell_text = np.array([row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()])
    table = ax.table(
        cellText=cell_text,
        cellLoc="center",
        rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
        rowLoc="center",
        colLabels=col_labels,
        colLoc="center",
        bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
    )
    for i in range(0, len(row_labels) + 1):
        for j in range(-1, len(col_labels)):
            if i == 0 and j == -1:
                continue
            elif i == 0:
                color = "xkcd:black"
            else:
                color = palette.get(row_labels[i - 1], "xkcd:black")
            cell = table[(i, j)]
            cell.set_text_props(weight="bold", color=color)
    table.auto_set_font_size(False)
    table.set_fontsize(7.5)
else:
    ax.set_xticks(
        ax.get_xticks(),
        labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
        rotation=0,
        ha="center",
    )
    if legend_kwargs:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles=handles, labels=labels, **legend_kwargs)


sns.despine(ax=ax)

filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
if include_table:
    filename_args += ["table"]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
output_filename = make_filename(*filename_args, filetype=filetype)
fig.savefig(output_dirpath / output_filename)
plt.show()
plt.close()
print(output_filename)
print(data["count"].sum(axis=1))

### Age

In [ ]:
subgroup = "Age"
title = subgroup
cut_kwargs = dict(
    bins=[0, 30, 50, df_metadata_genomics[("Metadata", subgroup)].max()],
    labels=["<30", "30 to 50", ">50"],
)
palette = {}
cmap_name = "Oranges_r"
n_rsids = 0

output_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "Correlations"
    / f"{main_data_type}_{other_data_type}"
    / subgroup.replace(" ", "")
)
output_dirpath.mkdir(parents=True, exist_ok=True)
for idx, rsid in enumerate(rsids):
    xlabel = rsid
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
    data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
    data = data.droplevel(level=0, axis=1).dropna()

    if cut_kwargs:
        data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
    data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
    data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
    data = data.sort_values(by=[rsid, subgroup])
    data = data.pivot(columns=subgroup, index=rsid)
    data = data.fillna(0)
    subgroups = list(data["count"].columns[::-1])
    # Get the colormap and select colors from the colormap
    # Values between 0 and 1 are used to select colors along the map
    if not palette and cmap_name:
        cmap = mpl.colormaps.get_cmap(cmap_name)
        colors = [
            cmap(x)
            for x in np.linspace(
                0 if cmap_name.endswith("_r") else 0.2,
                0.8 if cmap_name.endswith("_r") else 1,
                len(subgroups),
            )
        ]
        palette = dict(zip(subgroups, colors))
    ax = data["percentage"].plot(
        kind="bar",
        use_index=True,
        ax=ax,
        stacked=True,
        zorder=4,
        width=bar_width,
        xlabel=rsid,
        ylabel=None,
        legend=False,
        color=palette,
        edgecolor="black",
    )

    ax.set_ylim(0, 1.1)
    ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
    ax.set_ylabel(ylabel, fontdict=label_fontdict)
    ax.set_title(title, fontdict=title_fontdict)
    ax.yaxis.grid(**grid_kwargs)
    ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
    ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
    if include_table:
        ax.set_xticks([])
        ax.set_xlabel(None)
        ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

        row_labels = subgroups
        col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
        cell_text = np.array(
            [row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()]
        )
        table = ax.table(
            cellText=cell_text,
            cellLoc="center",
            rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
            rowLoc="center",
            colLabels=col_labels,
            colLoc="center",
            bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
        )
        for i in range(0, len(row_labels) + 1):
            for j in range(-1, len(col_labels)):
                if i == 0 and j == -1:
                    continue
                elif i == 0:
                    color = "xkcd:black"
                else:
                    color = palette.get(row_labels[i - 1], "xkcd:black")
                cell = table[(i, j)]
                cell.set_text_props(weight="bold", color=color)
        table.auto_set_font_size(False)
        table.set_fontsize(7.5)
    else:
        ax.set_xticks(
            ax.get_xticks(),
            labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
            rotation=0,
            ha="center",
        )
        if legend_kwargs:
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(handles=handles[::-1], labels=labels[::-1], **legend_kwargs)

    sns.despine(ax=ax)

    filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
    if include_table:
        filename_args += ["table"]
    if group_value != "ALL":
        filename_args += [f"{group_key}{group_value}"]
    output_filename = make_filename(*filename_args, filetype=filetype)
    fig.savefig(output_dirpath / output_filename)
    plt.show()
    plt.close()
    print(output_filename)
    print(data["count"].sum(axis=1))
    if n_rsids <= idx:
        break

In [ ]:
rsid = rsids[0]
subgroup = "Age"
title = subgroup
cut_kwargs = dict(
    bins=[0, 30, 50, df_metadata_genomics[("Metadata", subgroup)].max()],
    labels=["<30", "30 to 50", ">50"],
)
palette = {}
cmap_name = "Oranges_r"


output_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "Correlations"
    / f"{main_data_type}_{other_data_type}"
    / subgroup.replace(" ", "")
)
output_dirpath.mkdir(parents=True, exist_ok=True)
xlabel = rsid
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
data = data.droplevel(level=0, axis=1).dropna()

if cut_kwargs:
    data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
data = data.sort_values(by=[rsid, subgroup])
data = data.pivot(columns=subgroup, index=rsid)
data = data.fillna(0)
subgroups = list(data["count"].columns[::-1])
# Get the colormap and select colors from the colormap
# Values between 0 and 1 are used to select colors along the map
if not palette and cmap_name:
    cmap = mpl.colormaps.get_cmap(cmap_name)
    colors = [
        cmap(x)
        for x in np.linspace(
            0 if cmap_name.endswith("_r") else 0.2,
            0.8 if cmap_name.endswith("_r") else 1,
            len(subgroups),
        )
    ]
    palette = dict(zip(subgroups, colors))

ax = data["percentage"].plot(
    kind="bar",
    use_index=True,
    ax=ax,
    stacked=True,
    zorder=4,
    width=bar_width,
    xlabel=rsid,
    ylabel=None,
    legend=False,
    color=palette,
    edgecolor="black",
)

ax.set_ylim(0, 1.1)
ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
ax.set_ylabel(ylabel, fontdict=label_fontdict)
ax.set_title(title, fontdict=title_fontdict)
ax.yaxis.grid(**grid_kwargs)
ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
if include_table:
    ax.set_xticks([])
    ax.set_xlabel(None)
    ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

    row_labels = subgroups
    col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
    cell_text = np.array([row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()])
    table = ax.table(
        cellText=cell_text,
        cellLoc="center",
        rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
        rowLoc="center",
        colLabels=col_labels,
        colLoc="center",
        bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
    )
    for i in range(0, len(row_labels) + 1):
        for j in range(-1, len(col_labels)):
            if i == 0 and j == -1:
                continue
            elif i == 0:
                color = "xkcd:black"
            else:
                color = palette.get(row_labels[i - 1], "xkcd:black")
            cell = table[(i, j)]
            cell.set_text_props(weight="bold", color=color)
    table.auto_set_font_size(False)
    table.set_fontsize(7.5)
else:
    ax.set_xticks(
        ax.get_xticks(),
        labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
        rotation=0,
        ha="center",
    )
    if legend_kwargs:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles=handles, labels=labels, **legend_kwargs)


sns.despine(ax=ax)

filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
if include_table:
    filename_args += ["table"]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
output_filename = make_filename(*filename_args, filetype=filetype)
fig.savefig(output_dirpath / output_filename)
plt.show()
plt.close()
print(output_filename)
print(data["count"].sum(axis=1))

### BMI

In [ ]:
subgroup = "BMI"
title = subgroup
cut_kwargs = dict(
    bins=[0, 25, 30, 40, df_metadata_genomics[("Metadata", subgroup)].max()],
    labels=["<25", "25 to 30", "30 to 40", ">40"],
)
palette = {}
cmap_name = "Purples_r"
n_rsids = 0

output_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "Correlations"
    / f"{main_data_type}_{other_data_type}"
    / subgroup.replace(" ", "")
)
output_dirpath.mkdir(parents=True, exist_ok=True)
for idx, rsid in enumerate(rsids):
    xlabel = rsid
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
    data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
    data = data.droplevel(level=0, axis=1).dropna()

    if cut_kwargs:
        data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
    data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
    data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
    data = data.sort_values(by=[rsid, subgroup])
    data = data.pivot(columns=subgroup, index=rsid)
    data = data.fillna(0)
    subgroups = list(data["count"].columns[::-1])
    # Get the colormap and select colors from the colormap
    # Values between 0 and 1 are used to select colors along the map
    if not palette and cmap_name:
        cmap = mpl.colormaps.get_cmap(cmap_name)
        colors = [
            cmap(x)
            for x in np.linspace(
                0 if cmap_name.endswith("_r") else 0.2,
                0.8 if cmap_name.endswith("_r") else 1,
                len(subgroups),
            )
        ]
        palette = dict(zip(subgroups, colors))
    ax = data["percentage"].plot(
        kind="bar",
        use_index=True,
        ax=ax,
        stacked=True,
        zorder=4,
        width=bar_width,
        xlabel=rsid,
        ylabel=None,
        legend=False,
        color=palette,
        edgecolor="black",
    )

    ax.set_ylim(0, 1.1)
    ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
    ax.set_ylabel(ylabel, fontdict=label_fontdict)
    ax.set_title(title, fontdict=title_fontdict)
    ax.yaxis.grid(**grid_kwargs)
    ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
    ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
    if include_table:
        ax.set_xticks([])
        ax.set_xlabel(None)
        ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

        row_labels = subgroups
        col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
        cell_text = np.array(
            [row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()]
        )
        table = ax.table(
            cellText=cell_text,
            cellLoc="center",
            rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
            rowLoc="center",
            colLabels=col_labels,
            colLoc="center",
            bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
        )
        for i in range(0, len(row_labels) + 1):
            for j in range(-1, len(col_labels)):
                if i == 0 and j == -1:
                    continue
                elif i == 0:
                    color = "xkcd:black"
                else:
                    color = palette.get(row_labels[i - 1], "xkcd:black")
                cell = table[(i, j)]
                cell.set_text_props(weight="bold", color=color)
        table.auto_set_font_size(False)
        table.set_fontsize(7.5)
    else:
        ax.set_xticks(
            ax.get_xticks(),
            labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
            rotation=0,
            ha="center",
        )
        if legend_kwargs:
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(handles=handles[::-1], labels=labels[::-1], **legend_kwargs)

    sns.despine(ax=ax)

    filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
    if include_table:
        filename_args += ["table"]
    if group_value != "ALL":
        filename_args += [f"{group_key}{group_value}"]
    output_filename = make_filename(*filename_args, filetype=filetype)
    fig.savefig(output_dirpath / output_filename)
    plt.show()
    plt.close()
    print(output_filename)
    print(data["count"].sum(axis=1))

    if n_rsids <= idx:
        break

In [ ]:
rsid = rsids[0]
subgroup = "BMI"
title = subgroup
cut_kwargs = dict(
    bins=[0, 25, 30, 40, df_metadata_genomics[("Metadata", subgroup)].max()],
    labels=["<25", "25 to 30", "30 to 40", ">40"],
)
palette = {}
cmap_name = "Purples_r"


output_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "Correlations"
    / f"{main_data_type}_{other_data_type}"
    / subgroup.replace(" ", "")
)
output_dirpath.mkdir(parents=True, exist_ok=True)
xlabel = rsid
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
data = data.droplevel(level=0, axis=1).dropna()

if cut_kwargs:
    data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
data = data.sort_values(by=[rsid, subgroup])
data = data.pivot(columns=subgroup, index=rsid)
data = data.fillna(0)
subgroups = list(data["count"].columns[::-1])
# Get the colormap and select colors from the colormap
# Values between 0 and 1 are used to select colors along the map
if not palette and cmap_name:
    cmap = mpl.colormaps.get_cmap(cmap_name)
    colors = [
        cmap(x)
        for x in np.linspace(
            0 if cmap_name.endswith("_r") else 0.2,
            0.8 if cmap_name.endswith("_r") else 1,
            len(subgroups),
        )
    ]
    palette = dict(zip(subgroups, colors))

ax = data["percentage"].plot(
    kind="bar",
    use_index=True,
    ax=ax,
    stacked=True,
    zorder=4,
    width=bar_width,
    xlabel=rsid,
    ylabel=None,
    legend=False,
    color=palette,
    edgecolor="black",
)

ax.set_ylim(0, 1.1)
ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
ax.set_ylabel(ylabel, fontdict=label_fontdict)
ax.set_title(title, fontdict=title_fontdict)
ax.yaxis.grid(**grid_kwargs)
ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
if include_table:
    ax.set_xticks([])
    ax.set_xlabel(None)
    ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

    row_labels = subgroups
    col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
    cell_text = np.array([row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()])
    table = ax.table(
        cellText=cell_text,
        cellLoc="center",
        rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
        rowLoc="center",
        colLabels=col_labels,
        colLoc="center",
        bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
    )
    for i in range(0, len(row_labels) + 1):
        for j in range(-1, len(col_labels)):
            if i == 0 and j == -1:
                continue
            elif i == 0:
                color = "xkcd:black"
            else:
                color = palette.get(row_labels[i - 1], "xkcd:black")
            cell = table[(i, j)]
            cell.set_text_props(weight="bold", color=color)
    table.auto_set_font_size(False)
    table.set_fontsize(7.5)
else:
    ax.set_xticks(
        ax.get_xticks(),
        labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
        rotation=0,
        ha="center",
    )
    if legend_kwargs:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles=handles, labels=labels, **legend_kwargs)


sns.despine(ax=ax)

filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
if include_table:
    filename_args += ["table"]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
output_filename = make_filename(*filename_args, filetype=filetype)
fig.savefig(output_dirpath / output_filename)
plt.show()
plt.close()
print(output_filename)
print(data["count"].sum(axis=1))

In [ ]:
# subgroup = "Center"
# title = subgroup
# cut_kwargs = {}
# palette = {}
# cmap_name = "Oranges"
# n_rsids = 0


# output_dirpath = (
#     REPORTS_DIR
#     / "REDS"
#     / "Correlations"
#     / f"{main_data_type}_{other_data_type}"
#     / subgroup.replace(" ", "")
# )
# output_dirpath.mkdir(parents=True, exist_ok=True)
# for idx, rsid in enumerate(rsids):
#     xlabel = rsid
#     fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
#     data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
#     data = data.droplevel(level=0, axis=1).dropna()

#     if cut_kwargs:
#         data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
#     data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
#     data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
#     data = data.sort_values(by=[rsid, subgroup])
#     data = data.pivot(columns=subgroup, index=rsid)
#     data = data.fillna(0)
#     subgroups = list(data["count"].columns[::-1])
#     # Get the colormap and select colors from the colormap
#     # Values between 0 and 1 are used to select colors along the map
#     if not palette and cmap_name:
#         cmap = mpl.colormaps.get_cmap(cmap_name)
#         colors = [
#             cmap(x)
#             for x in np.linspace(
#                 0 if cmap_name.endswith("_r") else 0.2,
#                 0.8 if cmap_name.endswith("_r") else 1,
#                 len(subgroups),
#             )
#         ]
#         palette = dict(zip(subgroups, colors))
#     ax = data["percentage"].plot(
#         kind="bar",
#         use_index=True,
#         ax=ax,
#         stacked=True,
#         zorder=4,
#         width=bar_width,
#         xlabel=rsid,
#         ylabel=None,
#         legend=False,
#         color=palette,
#         edgecolor="black",
#     )

#     ax.set_ylim(0, 1.1)
#     ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
#     ax.set_ylabel(ylabel, fontdict=label_fontdict)
#     ax.set_title(title, fontdict=title_fontdict)
#     ax.yaxis.grid(**grid_kwargs)
#     ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
#     ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
#     ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
#     if include_table:
#         ax.set_xticks([])
#         ax.set_xlabel(None)
#         ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

#         row_labels = subgroups
#         col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
#         cell_text = np.array(
#             [row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()]
#         )
#         table = ax.table(
#             cellText=cell_text,
#             cellLoc="center",
#             rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
#             rowLoc="center",
#             colLabels=col_labels,
#             colLoc="center",
#             bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
#         )
#         for i in range(0, len(row_labels) + 1):
#             for j in range(-1, len(col_labels)):
#                 if i == 0 and j == -1:
#                     continue
#                 elif i == 0:
#                     color = "xkcd:black"
#                 else:
#                     color = palette.get(row_labels[i - 1], "xkcd:black")
#                 cell = table[(i, j)]
#                 cell.set_text_props(weight="bold", color=color)
#         table.auto_set_font_size(False)
#         table.set_fontsize(7.5)
#     else:
#         ax.set_xticks(
#             ax.get_xticks(),
#             labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
#             rotation=0,
#             ha="center",
#         )
#         if legend_kwargs:
#             handles, labels = ax.get_legend_handles_labels()
#             ax.legend(handles=handles[::-1], labels=labels[::-1], **legend_kwargs)

#     sns.despine(ax=ax)

#     filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
#     if include_table:
#         filename_args += ["table"]
#     if group_value != "ALL":
#         filename_args += [f"{group_key}{group_value}"]
#     output_filename = make_filename(*filename_args, filetype=filetype)
#     fig.savefig(output_dirpath / output_filename)
#     plt.show()
#     plt.close()
#     print(output_filename)
#     print(data["count"].sum(axis=1))
#     if n_rsids <= idx:
#         break

In [ ]:
# rsid = rsids[0]
# subgroup = "Center"
# title = subgroup
# cut_kwargs = {}
# palette = {}
# cmap_name = "Oranges"


# output_dirpath = (
#     REPORTS_DIR
#     / "REDS"
#     / "Correlations"
#     / f"{main_data_type}_{other_data_type}"
#     / subgroup.replace(" ", "")
# )
# output_dirpath.mkdir(parents=True, exist_ok=True)
# xlabel = rsid
# fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
# data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
# data = data.droplevel(level=0, axis=1).dropna()

# if cut_kwargs:
#     data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
# data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
# data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
# data = data.sort_values(by=[rsid, subgroup])
# data = data.pivot(columns=subgroup, index=rsid)
# data = data.fillna(0)
# subgroups = list(data["count"].columns[::-1])

# # Get the colormap and select colors from the colormap
# # Values between 0 and 1 are used to select colors along the map
# if not palette and cmap_name:
#     cmap = mpl.colormaps.get_cmap(cmap_name)
#     colors = [
#         cmap(x)
#         for x in np.linspace(
#             0 if cmap_name.endswith("_r") else 0.2,
#             0.8 if cmap_name.endswith("_r") else 1,
#             len(subgroups),
#         )
#     ]
#     palette = dict(zip(subgroups, colors))

# ax = data["percentage"].plot(
#     kind="bar",
#     use_index=True,
#     ax=ax,
#     stacked=True,
#     zorder=4,
#     width=bar_width,
#     xlabel=rsid,
#     ylabel=None,
#     legend=False,
#     color=palette,
#     edgecolor="black",
# )

# ax.set_ylim(0, 1.1)
# ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
# ax.set_ylabel(ylabel, fontdict=label_fontdict)
# ax.set_title(title, fontdict=title_fontdict)
# ax.yaxis.grid(**grid_kwargs)
# ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
# ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
# ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
# if include_table:
#     ax.set_xticks([])
#     ax.set_xlabel(None)
#     ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

#     row_labels = subgroups
#     col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
#     cell_text = np.array([row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()])
#     table = ax.table(
#         cellText=cell_text,
#         cellLoc="center",
#         rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
#         rowLoc="center",
#         colLabels=col_labels,
#         colLoc="center",
#         bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
#     )
#     for i in range(0, len(row_labels) + 1):
#         for j in range(-1, len(col_labels)):
#             if i == 0 and j == -1:
#                 continue
#             elif i == 0:
#                 color = "xkcd:black"
#             else:
#                 color = palette.get(row_labels[i - 1], "xkcd:black")
#             cell = table[(i, j)]
#             cell.set_text_props(weight="bold", color=color)
#     table.auto_set_font_size(False)
#     table.set_fontsize(7.5)
# else:
#     ax.set_xticks(
#         ax.get_xticks(),
#         labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
#         rotation=0,
#         ha="center",
#     )
#     if legend_kwargs:
#         handles, labels = ax.get_legend_handles_labels()
#         ax.legend(handles=handles, labels=labels, **legend_kwargs)


# sns.despine(ax=ax)

# filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
# if include_table:
#     filename_args += ["table"]
# if group_value != "ALL":
#     filename_args += [f"{group_key}{group_value}"]
# output_filename = make_filename(*filename_args, filetype=filetype)
# fig.savefig(output_dirpath / output_filename)
# plt.show()
# plt.close()
# print(output_filename)
# print(data["count"].sum(axis=1))

### Blood group

In [ ]:
# subgroup = "Blood.Group.ABO"
# title = subgroup
# cut_kwargs = {}
# palette = {}
# cmap_name = "Reds"
# n_rsids = 0

# output_dirpath = (
#     REPORTS_DIR
#     / "REDS"
#     / "Correlations"
#     / f"{main_data_type}_{other_data_type}"
#     / subgroup.replace(" ", "")
# )
# output_dirpath.mkdir(parents=True, exist_ok=True)
# for idx, rsid in enumerate(rsids):
#     xlabel = rsid
#     fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
#     data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
#     data = data.droplevel(level=0, axis=1).dropna()

#     if cut_kwargs:
#         data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
#     data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
#     data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
#     data = data.sort_values(by=[rsid, subgroup])
#     data = data.pivot(columns=subgroup, index=rsid)
#     data = data.fillna(0)
#     subgroups = list(data["count"].columns[::-1])

#     # Get the colormap and select colors from the colormap
#     # Values between 0 and 1 are used to select colors along the map
#     if not palette and cmap_name:
#         cmap = mpl.colormaps.get_cmap(cmap_name)
#         colors = [
#             cmap(x)
#             for x in np.linspace(
#                 0 if cmap_name.endswith("_r") else 0.2,
#                 0.8 if cmap_name.endswith("_r") else 1,
#                 len(subgroups),
#             )
#         ]
#         palette = dict(zip(subgroups, colors))
#     ax = data["percentage"].plot(
#         kind="bar",
#         use_index=True,
#         ax=ax,
#         stacked=True,
#         zorder=4,
#         width=bar_width,
#         xlabel=rsid,
#         ylabel=None,
#         legend=False,
#         color=palette,
#         edgecolor="black",
#     )

#     ax.set_ylim(0, 1.1)
#     ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
#     ax.set_ylabel(ylabel, fontdict=label_fontdict)
#     ax.set_title(title, fontdict=title_fontdict)
#     ax.yaxis.grid(**grid_kwargs)
#     ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
#     ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
#     ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
#     if include_table:
#         ax.set_xticks([])
#         ax.set_xlabel(None)
#         ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

#         row_labels = subgroups
#         col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
#         cell_text = np.array(
#             [row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()]
#         )
#         table = ax.table(
#             cellText=cell_text,
#             cellLoc="center",
#             rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
#             rowLoc="center",
#             colLabels=col_labels,
#             colLoc="center",
#             bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
#         )
#         for i in range(0, len(row_labels) + 1):
#             for j in range(-1, len(col_labels)):
#                 if i == 0 and j == -1:
#                     continue
#                 elif i == 0:
#                     color = "xkcd:black"
#                 else:
#                     color = palette.get(row_labels[i - 1], "xkcd:black")
#                 cell = table[(i, j)]
#                 cell.set_text_props(weight="bold", color=color)
#         table.auto_set_font_size(False)
#         table.set_fontsize(7.5)
#     else:
#         ax.set_xticks(
#             ax.get_xticks(),
#             labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
#             rotation=0,
#             ha="center",
#         )
#         if legend_kwargs:
#             handles, labels = ax.get_legend_handles_labels()
#             ax.legend(handles=handles[::-1], labels=labels[::-1], **legend_kwargs)

#     sns.despine(ax=ax)

#     filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
#     if include_table:
#         filename_args += ["table"]
#     if group_value != "ALL":
#         filename_args += [f"{group_key}{group_value}"]
#     output_filename = make_filename(*filename_args, filetype=filetype)
#     fig.savefig(output_dirpath / output_filename)
#     plt.show()
#     plt.close()
#     print(output_filename)
#     print(data["count"].sum(axis=1))
#     if n_rsids <= idx:
#         break

In [ ]:
# rsid = rsids[0]
# subgroup = "Blood.Group.ABO"
# title = subgroup
# cut_kwargs = {}
# palette = {}
# cmap_name = "Reds"
# n_rsids = 0

# output_dirpath = (
#     REPORTS_DIR
#     / "REDS"
#     / "Correlations"
#     / f"{main_data_type}_{other_data_type}"
#     / subgroup.replace(" ", "")
# )
# output_dirpath.mkdir(parents=True, exist_ok=True)
# xlabel = rsid
# fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
# data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
# data = data.droplevel(level=0, axis=1).dropna()

# if cut_kwargs:
#     data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
# data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
# data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
# data = data.sort_values(by=[rsid, subgroup])
# data = data.pivot(columns=subgroup, index=rsid)
# data = data.fillna(0)
# subgroups = list(data["count"].columns[::-1])

# # Get the colormap and select colors from the colormap
# # Values between 0 and 1 are used to select colors along the map
# if not palette and cmap_name:
#     cmap = mpl.colormaps.get_cmap(cmap_name)
#     colors = [
#         cmap(x)
#         for x in np.linspace(
#             0 if cmap_name.endswith("_r") else 0.2,
#             0.8 if cmap_name.endswith("_r") else 1,
#             len(subgroups),
#         )
#     ]
#     palette = dict(zip(subgroups, colors))

# ax = data["percentage"].plot(
#     kind="bar",
#     use_index=True,
#     ax=ax,
#     stacked=True,
#     zorder=4,
#     width=bar_width,
#     xlabel=rsid,
#     ylabel=None,
#     legend=False,
#     color=palette,
#     edgecolor="black",
# )

# ax.set_ylim(0, 1.1)
# ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
# ax.set_ylabel(ylabel, fontdict=label_fontdict)
# ax.set_title(title, fontdict=title_fontdict)
# ax.yaxis.grid(**grid_kwargs)
# ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
# ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
# ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
# if include_table:
#     ax.set_xticks([])
#     ax.set_xlabel(None)
#     ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

#     row_labels = subgroups
#     col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
#     cell_text = np.array([row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()])
#     table = ax.table(
#         cellText=cell_text,
#         cellLoc="center",
#         rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
#         rowLoc="center",
#         colLabels=col_labels,
#         colLoc="center",
#         bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
#     )
#     for i in range(0, len(row_labels) + 1):
#         for j in range(-1, len(col_labels)):
#             if i == 0 and j == -1:
#                 continue
#             elif i == 0:
#                 color = "xkcd:black"
#             else:
#                 color = palette.get(row_labels[i - 1], "xkcd:black")
#             cell = table[(i, j)]
#             cell.set_text_props(weight="bold", color=color)
#     table.auto_set_font_size(False)
#     table.set_fontsize(7.5)
# else:
#     ax.set_xticks(
#         ax.get_xticks(),
#         labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
#         rotation=0,
#         ha="center",
#     )
#     if legend_kwargs:
#         handles, labels = ax.get_legend_handles_labels()
#         ax.legend(handles=handles, labels=labels, **legend_kwargs)


# sns.despine(ax=ax)

# filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
# if include_table:
#     filename_args += ["table"]
# if group_value != "ALL":
#     filename_args += [f"{group_key}{group_value}"]
# output_filename = make_filename(*filename_args, filetype=filetype)
# fig.savefig(output_dirpath / output_filename)
# plt.show()
# plt.close()
# print(output_filename)
# print(data["count"].sum(axis=1))

### Additive

In [ ]:
# subgroup = "AS"
# title = subgroup
# cut_kwargs = {}
# palette = {}
# cmap_name = "coolwarm"
# n_rsids = 0

# output_dirpath = (
#     REPORTS_DIR
#     / "REDS"
#     / "Correlations"
#     / f"{main_data_type}_{other_data_type}"
#     / subgroup.replace(" ", "")
# )
# output_dirpath.mkdir(parents=True, exist_ok=True)
# for idx, rsid in enumerate(rsids):
#     xlabel = rsid
#     fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
#     data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
#     data = data.droplevel(level=0, axis=1).dropna()

#     if cut_kwargs:
#         data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
#     data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
#     data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
#     data = data.sort_values(by=[rsid, subgroup])
#     data = data.pivot(columns=subgroup, index=rsid)
#     data = data.fillna(0)
#     subgroups = list(data["count"].columns[::-1])

#     # Get the colormap and select colors from the colormap
#     # Values between 0 and 1 are used to select colors along the map
#     if not palette and cmap_name:
#         cmap = mpl.colormaps.get_cmap(cmap_name)
#         colors = [
#             cmap(x)
#             for x in np.linspace(
#                 0 if cmap_name.endswith("_r") else 0.2,
#                 0.8 if cmap_name.endswith("_r") else 1,
#                 len(subgroups),
#             )
#         ]
#         palette = dict(zip(subgroups, colors))
#     ax = data["percentage"].plot(
#         kind="bar",
#         use_index=True,
#         ax=ax,
#         stacked=True,
#         zorder=4,
#         width=bar_width,
#         xlabel=rsid,
#         ylabel=None,
#         legend=False,
#         color=palette,
#         edgecolor="black",
#     )

#     ax.set_ylim(0, 1.1)
#     ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
#     ax.set_ylabel(ylabel, fontdict=label_fontdict)
#     ax.set_title(title, fontdict=title_fontdict)
#     ax.yaxis.grid(**grid_kwargs)
#     ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
#     ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
#     ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
#     if include_table:
#         ax.set_xticks([])
#         ax.set_xlabel(None)
#         ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

#         row_labels = subgroups
#         col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
#         cell_text = np.array(
#             [row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()]
#         )
#         table = ax.table(
#             cellText=cell_text,
#             cellLoc="center",
#             rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
#             rowLoc="center",
#             colLabels=col_labels,
#             colLoc="center",
#             bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
#         )
#         for i in range(0, len(row_labels) + 1):
#             for j in range(-1, len(col_labels)):
#                 if i == 0 and j == -1:
#                     continue
#                 elif i == 0:
#                     color = "xkcd:black"
#                 else:
#                     color = palette.get(row_labels[i - 1], "xkcd:black")
#                 cell = table[(i, j)]
#                 cell.set_text_props(weight="bold", color=color)
#         table.auto_set_font_size(False)
#         table.set_fontsize(7.5)
#     else:
#         ax.set_xticks(
#             ax.get_xticks(),
#             labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
#             rotation=0,
#             ha="center",
#         )
#         if legend_kwargs:
#             handles, labels = ax.get_legend_handles_labels()
#             ax.legend(handles=handles[::-1], labels=labels[::-1], **legend_kwargs)

#     sns.despine(ax=ax)

#     filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
#     if include_table:
#         filename_args += ["table"]
#     if group_value != "ALL":
#         filename_args += [f"{group_key}{group_value}"]
#     output_filename = make_filename(*filename_args, filetype=filetype)
#     fig.savefig(output_dirpath / output_filename)
#     plt.show()
#     plt.close()
#     print(output_filename)
#     print(data["count"].sum(axis=1))
#     if n_rsids <= idx:
#         break

In [ ]:
# rsid = rsids[0]
# subgroup = "AS"
# title = subgroup
# cut_kwargs = {}
# palette = {}
# cmap_name = "coolwarm"
# n_rsids = 0

# output_dirpath = (
#     REPORTS_DIR
#     / "REDS"
#     / "Correlations"
#     / f"{main_data_type}_{other_data_type}"
#     / subgroup.replace(" ", "")
# )
# output_dirpath.mkdir(parents=True, exist_ok=True)
# xlabel = rsid
# fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
# data = df_metadata_genomics.loc[:, [(f"Genomics-{GENE_ID}", rsid), ("Metadata", subgroup)]]
# data = data.droplevel(level=0, axis=1).dropna()

# if cut_kwargs:
#     data[subgroup] = pd.cut(data[subgroup], **cut_kwargs)
# data = data.groupby([rsid, subgroup], as_index=False, observed=True).value_counts()
# data["percentage"] = data["count"] / data.groupby(rsid)["count"].transform("sum")
# data = data.sort_values(by=[rsid, subgroup])
# data = data.pivot(columns=subgroup, index=rsid)
# data = data.fillna(0)
# subgroups = list(data["count"].columns[::-1])

# # Get the colormap and select colors from the colormap
# # Values between 0 and 1 are used to select colors along the map
# if not palette and cmap_name:
#     cmap = mpl.colormaps.get_cmap(cmap_name)
#     colors = [
#         cmap(x)
#         for x in np.linspace(
#             0 if cmap_name.endswith("_r") else 0.2,
#             0.8 if cmap_name.endswith("_r") else 1,
#             len(subgroups),
#         )
#     ]
#     palette = dict(zip(subgroups, colors))

# ax = data["percentage"].plot(
#     kind="bar",
#     use_index=True,
#     ax=ax,
#     stacked=True,
#     zorder=4,
#     width=bar_width,
#     xlabel=rsid,
#     ylabel=None,
#     legend=False,
#     color=palette,
#     edgecolor="black",
# )

# ax.set_ylim(0, 1.1)
# ax.set_xlabel(xlabel if not include_table else None, fontdict=label_fontdict)
# ax.set_ylabel(ylabel, fontdict=label_fontdict)
# ax.set_title(title, fontdict=title_fontdict)
# ax.yaxis.grid(**grid_kwargs)
# ax.set_yticks(np.arange(0, 1 + tick_step, tick_step), minor=False)
# ax.set_yticks(np.arange(tick_step / 2, 1 + tick_step / 2, tick_step), minor=True)
# ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
# if include_table:
#     ax.set_xticks([])
#     ax.set_xlabel(None)
#     ax.set_title("\n".join([rsid, subgroup]), fontdict=title_fontdict)

#     row_labels = subgroups
#     col_labels = [x.replace(" ", "\n") for x in allele_name_map.values()]
#     cell_text = np.array([row.values.astype(int) for i, row in data["count"].T[::-1].iterrows()])
#     table = ax.table(
#         cellText=cell_text,
#         cellLoc="center",
#         rowLabels=[{x: x.split("_")[0] for x in subgroups}.get(x, x) for x in row_labels],
#         rowLoc="center",
#         colLabels=col_labels,
#         colLoc="center",
#         bbox=(0, -0.1 * (len(row_labels) + 1), 1, 0.1 * (len(row_labels) + 1)),
#     )
#     for i in range(0, len(row_labels) + 1):
#         for j in range(-1, len(col_labels)):
#             if i == 0 and j == -1:
#                 continue
#             elif i == 0:
#                 color = "xkcd:black"
#             else:
#                 color = palette.get(row_labels[i - 1], "xkcd:black")
#             cell = table[(i, j)]
#             cell.set_text_props(weight="bold", color=color)
#     table.auto_set_font_size(False)
#     table.set_fontsize(7.5)
# else:
#     ax.set_xticks(
#         ax.get_xticks(),
#         labels=[allele_name_map[x].replace(" ", "\n") for x in ax.get_xticks()],
#         rotation=0,
#         ha="center",
#     )
#     if legend_kwargs:
#         handles, labels = ax.get_legend_handles_labels()
#         ax.legend(handles=handles, labels=labels, **legend_kwargs)


# sns.despine(ax=ax)

# filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
# if include_table:
#     filename_args += ["table"]
# if group_value != "ALL":
#     filename_args += [f"{group_key}{group_value}"]
# output_filename = make_filename(*filename_args, filetype=filetype)
# fig.savefig(output_dirpath / output_filename)
# plt.show()
# plt.close()
# print(output_filename)
# print(data["count"].sum(axis=1))